# Oxideon Quickstart

Run **Oxideon** (`pip install oxideon`) directly in Google Colab or Jupyter:
- Zero-copy strided views & higher-order autograd (`create_graph=True`)
- Training a neural network with `AdamW` & `LayerNorm`
- Zero-copy NumPy / DLPack interop & Einops `rearrange`
- Quick CPU speed check vs PyTorch

Repo: https://github.com/voidstackloop/oxideon

In [ ]:
!pip install -q oxideon

In [ ]:
import oxideon as ox

# 1. Zero-copy strided views
x = ox.Tensor.arange(0.0, 24.0, 1.0).reshape([2, 3, 4])
v = x.transpose(0, 2)
assert v.shares_storage(x)
print("v shape:", list(v.shape), "shares_storage:", v.shares_storage(x))

# 2. Einops-style rearrange & channels_last
img = ox.randn([2, 8, 4, 4])
nhwc = img.rearrange("b c h w -> b h w c")
cl = img.to_channels_last()
print("rearranged shape:", list(nhwc.shape), "is_channels_last:", cl.is_channels_last())

In [ ]:
# 3. Higher-order autograd (1st, 2nd, and 3rd derivatives)
x = ox.Tensor.from_slice([3.0], [1], requires_grad=True)
y = x.powf(4.0)
dy = ox.grad([y], [x], create_graph=True)[0]
d2y = ox.grad([dy], [x], create_graph=True)[0]
d3y = ox.grad([d2y], [x])[0]
print("y=x^4 at x=3 -> dy:", dy.to_list(), "d2y:", d2y.to_list(), "d3y:", d3y.to_list())

In [ ]:
# 4. Train a compact MLP with AdamW
class MLP(ox.nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.fc1 = ox.nn.Linear(16, d)
        self.ln = ox.nn.LayerNorm(d)
        self.fc2 = ox.nn.Linear(d, 4)

    def forward(self, x):
        return self.fc2(self.ln(self.fc1(x)).gelu())

model = MLP()
opt = ox.optim.AdamW(model.parameters(), lr=1e-2)
xs = ox.randn([32, 16])
ys = ox.randn([32, 4])

for step in range(40):
    opt.zero_grad()
    pred = model(xs)
    loss = ((pred - ys) * (pred - ys)).mean()
    loss.backward()
    opt.step()
    if step in (0, 19, 39):
        print(f"step {step:2d}: loss = {loss.item():.6f}")

In [ ]:
# 5. Zero-copy NumPy / DLPack interop & quick CPU benchmark vs PyTorch
import time

x_np = ox.randn([32, 128, 768]).numpy()
xo = ox.from_numpy(x_np)
for _ in range(3):
    _ = xo.softmax(-1)
t0 = time.perf_counter()
for _ in range(20):
    _ = xo.softmax(-1)
ox_ms = (time.perf_counter() - t0) / 20 * 1000

try:
    import torch
    xt = torch.utils.dlpack.from_dlpack(xo)
    for _ in range(3):
        _ = torch.softmax(xt, -1)
    t0 = time.perf_counter()
    for _ in range(20):
        _ = torch.softmax(xt, -1)
    pt_ms = (time.perf_counter() - t0) / 20 * 1000
    print(f"softmax [32,128,768]: Oxideon={ox_ms:.2f} ms, PyTorch={pt_ms:.2f} ms")
except ImportError:
    print(f"softmax [32,128,768]: Oxideon={ox_ms:.2f} ms (install torch to compare)")